# Ames Mutagenicity: Classical and Graph Neural Network Models

## Part 15 — Intervals for the regime decomposition

The decomposition by alert regime became the central result of the attribution analysis in
the last revision, and it was reported as twelve point estimates with no intervals. That is
the error the paired resampling was introduced to correct, displaced from one claim to
another: the aggregate comparison now carries its uncertainty and the decomposition does not.

This notebook applies the same paired resampling of molecules and models to each regime, so
that the statement about where a trained model exceeds the baseline and where it falls below
carries an interval. Nothing is trained; the stored attribution scores are reused.

**Contents**

1. Environment and stored quantities
2. The regime decomposition with intervals
3. Summary

*Approximate runtime: twenty minutes on CPU.*

## 1. Environment and stored quantities

In [1]:
!pip install -q rdkit
!pip install -q torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 21.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 16.3 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = "/content/drive/MyDrive/ames_pipeline"
DATADIR, R, EXPL, MODELS = f"{WORK}/data", f"{WORK}/results", f"{WORK}/explain", f"{WORK}/models"
os.makedirs(R, exist_ok=True)
print("results:", R)

Mounted at /content/drive
results: /content/drive/MyDrive/ames_pipeline/results


In [3]:
import sys, json, glob, time
import numpy as np, pandas as pd
from collections import defaultdict
from scipy import stats
from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
RDLogger.DisableLog("rdApp.*")

SEED, PRIMARY_SPLIT_SEED = 42, 101
K_FRAC, K_GRID, N_TIEBREAK, N_BOOT = 0.30, [0.10,0.20,0.30,0.40,0.50], 200, 10000

def safe_scaffold(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return ""
    try: return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except Exception:
        try: return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))
        except Exception: return smi

def make_scaffold_split(d, seed, frac=(0.8, 0.1, 0.1)):
    groups = defaultdict(list)
    for i, smi in zip(d.index, d["canonical"].values):
        groups[safe_scaffold(smi)].append(i)
    gs = list(groups.values()); rng = np.random.default_rng(seed)
    gs.sort(key=lambda g: (-len(g), rng.random()))
    n = len(d); a, b = int(frac[0]*n), int((frac[0]+frac[1])*n)
    tr, va, te = [], [], []
    for g in gs:
        if len(tr)+len(g) <= a: tr += g
        elif len(tr)+len(va)+len(g) <= b: va += g
        else: te += g
    return d.loc[tr], d.loc[va], d.loc[te]

df = pd.concat([pd.read_csv(f"{DATADIR}/{f}") for f in ["train.csv","valid.csv","test.csv"]],
               ignore_index=True)[["Drug_ID","smiles","label"]]
df["canonical"] = df["smiles"].apply(
    lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) else None)
df = df.dropna(subset=["canonical"]).drop_duplicates(
    subset="canonical", keep="first").reset_index(drop=True)
_, _, df_test = [x.reset_index(drop=True) for x in make_scaffold_split(df, PRIMARY_SPLIT_SEED)]
assert set(df_test["canonical"]) == set(pd.read_csv(f"{R}/primary_test.csv")["canonical"])

TOX = {"aromatic_nitro":"[a][N+](=O)[O-]","aromatic_amine":"[a][NX3;H2,H1;!$(N[!#6]);!$(N=*)]",
 "aromatic_nitroso":"[a][NX2]=[OX1]","alkyl_nitrite":"[CX4][OX2][NX2]=[OX1]",
 "nitrosamine":"[NX3][NX2]=[OX1]","epoxide":"C1OC1","aziridine":"C1NC1",
 "azide":"[NX2]=[NX2+]=[NX1-]","azo_aromatic":"[a][NX2]=[NX2][a]",
 "aliphatic_halide":"[CX4][F,Cl,Br,I]","hetero_hetero_bond":"[N,O;H1,H2][N,O;H1,H2]",
 "polycyclic_aromatic":"c1ccc2ccccc2c1","aromatic_hydroxylamine":"[a][NX3;H1][OX2;H1]",
 "michael_acceptor":"[CX3]=[CX3][CX3]=[OX1]","aromatic_azoxy":"[a][NX2]=[NX3+]([O-])[a]",
 "nitrogen_mustard":"[NX3]([CX4][CX4][F,Cl,Br,I])[CX4][CX4][F,Cl,Br,I]",
 "aromatic_hydrazine":"[a][NX3;H1][NX3;H2]","alkyl_sulfonate":"[CX4][OX2][SX4](=[OX1])(=[OX1])"}
PAT = {k: Chem.MolFromSmarts(v) for k, v in TOX.items()}
REGIMES = {"full": list(TOX), "no_polycyclic": [k for k in TOX if k != "polycyclic_aromatic"],
           "polycyclic_only": ["polycyclic_aromatic"]}

def build_eval(frame, alerts):
    pats = {k: PAT[k] for k in alerts}; ev = []
    for smi, lab in zip(frame["canonical"], frame["label"]):
        if lab != 1: continue
        mol = Chem.MolFromSmiles(smi)
        if mol is None or mol.GetNumAtoms() == 0: continue
        t = set()
        for p in pats.values():
            for mt in mol.GetSubstructMatches(p): t.update(mt)
        if not t or len(t) == mol.GetNumAtoms(): continue
        ev.append({"smiles": smi, "mol": mol, "tox": t, "n_atoms": mol.GetNumAtoms()})
    return ev, np.array([len(e["tox"])/e["n_atoms"] for e in ev])

EVAL, CHANCE = build_eval(df_test, REGIMES["full"])
N_EVAL = len(EVAL)
DEG = [np.array([a.GetDegree() for a in e["mol"].GetAtoms()], dtype=float) for e in EVAL]
SMI2I = {e["smiles"]: i for i, e in enumerate(EVAL)}
print(f"evaluation set {N_EVAL}, chance {CHANCE.mean():.4f}")

def top_k(scores, k_frac, tie_seed):
    """Index sets of the highest-ranked k fraction, one per molecule."""
    rng = np.random.default_rng(tie_seed); out = []
    for e, sc in zip(EVAL, scores):
        n = e["n_atoms"]; k = max(1, int(round(k_frac*n)))
        order = np.lexsort((rng.random(n), -np.asarray(sc, dtype=float)))
        out.append(set(order[:k].tolist()))
    return out

def overlap_vec(scores, k_frac=K_FRAC, tie_seed=0, ev=None, ch=None):
    ev = EVAL if ev is None else ev
    rng = np.random.default_rng(tie_seed); out = np.full(len(ev), np.nan)
    for i, (e, sc) in enumerate(zip(ev, scores)):
        if sc is None or len(sc) != e["n_atoms"]: continue
        n = e["n_atoms"]; k = max(1, int(round(k_frac*n)))
        order = np.lexsort((rng.random(n), -np.asarray(sc, dtype=float)))
        out[i] = len(set(order[:k].tolist()) & e["tox"])/k
    return out

def mean_overlap(scores, k_frac=K_FRAC, n=N_TIEBREAK, base=0, ev=None):
    return np.nanmean([overlap_vec(scores, k_frac, base+j, ev) for j in range(n)], axis=0)

def lift_of(ov, ch=None):
    ch = CHANCE if ch is None else ch
    m = ~np.isnan(ov)
    return float(ov[m].mean()/ch[m].mean()) if m.sum() else np.nan

evaluation set 237, chance 0.4360


In [4]:
# per-molecule overlaps for the degree ranking, at every retained fraction
DEG_OV = {k: mean_overlap(DEG, k, base=4000+int(k*100)) for k in K_GRID}
print("degree lift by fraction:",
      {k: round(lift_of(DEG_OV[k]), 4) for k in K_GRID})

# per-molecule overlaps for every trained model, from the stored attribution scores
def load_arch(arch):
    pat = (f"{EXPL}/scores_ig_seed*.npz" if arch == "gin"
           else f"{EXPL}/scores_ig_{arch}_seed*.npz")
    files = sorted(glob.glob(pat))
    if arch == "gin":
        files = [f for f in files if "_gat_" not in f and "_gcn_" not in f
                 and "no_degree" not in f]
    out = {k: [] for k in K_GRID}; seeds = []
    for f in files:
        z = np.load(f); sc = [z[f"m{i}"] for i in range(N_EVAL)]
        for k in K_GRID:
            out[k].append(mean_overlap(sc, k, n=50, base=hash(f) % 10000))
        seeds.append(int(f.split("seed")[-1].split(".")[0]))
    return {k: np.array(v) for k, v in out.items()}, seeds

MODEL_OV, SEEDS = {}, {}
t0 = time.time()
for arch in ["gin", "gat", "gcn"]:
    MODEL_OV[arch], SEEDS[arch] = load_arch(arch)
    print(f"  {arch}: {len(SEEDS[arch])} models "
          f"({MODEL_OV[arch][K_FRAC].shape}) ({time.time()-t0:.0f} s)")
assert all(len(SEEDS[a]) >= 15 for a in SEEDS), "attribution scores missing; rerun Part 12"

degree lift by fraction: {0.1: 1.2015, 0.2: 1.2183, 0.3: 1.2137, 0.4: 1.1678, 0.5: 1.1084}
  gin: 20 models ((20, 237)) (16 s)
  gat: 20 models ((20, 237)) (30 s)
  gcn: 20 models ((20, 237)) (43 s)


## 2. The regime decomposition with intervals

Each regime defines its own evaluation set, its own chance level and its own attainable
ceiling. Molecules that a reduced library admits but the designated evaluation set excluded
carry no stored scores and are set aside, so that every arm of a regime is scored on the same
molecules. Six differences are tested, three architectures in each of the two reduced
regimes, and Holm's correction is applied across them.

In [5]:
def restrict(ev, ch):
    keep = [i for i, e in enumerate(ev) if e["smiles"] in SMI2I]
    return [ev[i] for i in keep], ch[keep], len(ev) - len(keep)

def paired_regime(model_ov, deg_ov, ch, n_boot=N_BOOT, seed=SEED):
    """(architecture lift - degree lift) with molecules and models resampled together."""
    rng = np.random.default_rng(seed)
    n_mol = model_ov.shape[1]; n_mod = model_ov.shape[0]
    out = np.empty(n_boot)
    for b in range(n_boot):
        mi = rng.integers(0, n_mol, n_mol); si = rng.integers(0, n_mod, n_mod)
        mo = np.nanmean(model_ov[np.ix_(si, mi)], axis=0); c = ch[mi]
        m1 = ~np.isnan(mo); m2 = ~np.isnan(deg_ov[mi])
        out[b] = (mo[m1].mean()/c[m1].mean()) - (deg_ov[mi][m2].mean()/c[m2].mean())
    return out

files = {"gin": [f for f in sorted(glob.glob(f"{EXPL}/scores_ig_seed*.npz"))
                 if "_gat_" not in f and "_gcn_" not in f and "no_degree" not in f],
         "gat": sorted(glob.glob(f"{EXPL}/scores_ig_gat_seed*.npz")),
         "gcn": sorted(glob.glob(f"{EXPL}/scores_ig_gcn_seed*.npz"))}

rows = []
for reg, alerts in REGIMES.items():
    ev0, ch0 = build_eval(df_test, alerts)
    ev, ch, dropped = restrict(ev0, ch0)
    orc = [np.array([1.0 if a in e["tox"] else 0.0 for a in range(e["n_atoms"])]) for e in ev]
    ceil = lift_of(mean_overlap(orc, n=1, base=0, ev=ev), ch)
    dg = [np.array([a.GetDegree() for a in e["mol"].GetAtoms()], dtype=float) for e in ev]
    d_ov = mean_overlap(dg, base=16000, ev=ev)
    d_lift = lift_of(d_ov, ch)
    for arch in ["gin", "gat", "gcn"]:
        ov = np.array([mean_overlap([np.load(f)[f"m{SMI2I[e['smiles']]}"] for e in ev],
                                    n=50, base=17000, ev=ev) for f in files[arch]])
        d = paired_regime(ov, d_ov, ch, seed=SEED + hash(reg) % 1000)
        m_lift = lift_of(np.nanmean(ov, axis=0), ch)
        lo, hi = np.percentile(d, [2.5, 97.5])
        rows.append({"regime": reg, "architecture": arch.upper(), "n_eval": len(ev),
                     "n_dropped": dropped, "n_models": len(files[arch]),
                     "ceiling": ceil, "degree_lift": d_lift, "model_lift": m_lift,
                     "difference": m_lift - d_lift, "ci_low": lo, "ci_high": hi,
                     "p_two_sided": 2*min((d > 0).mean(), (d < 0).mean()),
                     "degree_pct_of_range": 100*(d_lift-1)/(ceil-1),
                     "model_pct_of_range": 100*(m_lift-1)/(ceil-1)})
    print(f"  {reg}: {len(ev)} molecules"
          + (f", {dropped} set aside" if dropped else ""))

regime = pd.DataFrame(rows)
red = regime.regime != "full"          # correction across the two reduced regimes
p = regime.loc[red, "p_two_sided"].values
order = np.argsort(p); k = len(p); run = 0.0; adj = np.empty(k)
for r_, i_ in enumerate(order):
    run = max(run, (k - r_)*p[i_]); adj[i_] = min(run, 1.0)
regime["p_holm"] = np.nan
regime.loc[red, "p_holm"] = adj
regime["resolved"] = regime.p_holm < 0.05
regime.to_csv(f"{R}/table76_regime_paired.csv", index=False)
print()
print(regime[["regime","architecture","degree_lift","model_lift","difference",
              "ci_low","ci_high","p_holm","resolved"]].round(4).to_string(index=False))

  full: 237 molecules
  no_polycyclic: 174 molecules, 7 set aside
  polycyclic_only: 97 molecules, 7 set aside

         regime architecture  degree_lift  model_lift  difference  ci_low  ci_high  p_holm  resolved
           full          GIN       1.2128      1.2655      0.0527 -0.0134   0.1199     NaN     False
           full          GAT       1.2128      1.2026     -0.0102 -0.0455   0.0253     NaN     False
           full          GCN       1.2128      1.1531     -0.0597 -0.1143  -0.0062     NaN     False
  no_polycyclic          GIN       1.2954      1.7375      0.4421  0.3042   0.5880  0.0000      True
  no_polycyclic          GAT       1.2954      1.3535      0.0581 -0.0363   0.1547  0.4624     False
  no_polycyclic          GCN       1.2954      1.1836     -0.1118 -0.2355   0.0201  0.2872     False
polycyclic_only          GIN       1.1716      0.9410     -0.2306 -0.3252  -0.1443  0.0000      True
polycyclic_only          GAT       1.1716      1.1094     -0.0622 -0.1352   0.00

## 3. Summary

In [6]:
print("percentage of the attainable above-chance range")
piv = regime.pivot_table(index="architecture", columns="regime", values="model_pct_of_range")
deg = regime.groupby("regime").degree_pct_of_range.first()
print(piv[["full","no_polycyclic","polycyclic_only"]].round(1).to_string())
print("  atom degree" + "".join(f"{deg[c]:>17.1f}" for c in
      ["full","no_polycyclic","polycyclic_only"]))
print()
for reg in ["no_polycyclic", "polycyclic_only"]:
    s = regime[regime.regime == reg]
    n = int(s.resolved.sum())
    print(f"  {reg}: {n} of {len(s)} architectures separated from the baseline after correction")
    for _, r in s.iterrows():
        mark = "*" if r.resolved else " "
        print(f"    {mark} {r.architecture}: {r.difference:+.3f} "
              f"[{r.ci_low:+.3f}, {r.ci_high:+.3f}]  p = {r.p_holm:.3f}")
print("\nWritten to", R)
print("  - table76_regime_paired.csv")
import shutil
from google.colab import files
shutil.make_archive("/content/part15_results", "zip", R)
files.download("/content/part15_results.zip")

percentage of the attainable above-chance range
regime        full  no_polycyclic  polycyclic_only
architecture                                      
GAT           22.2           19.3             19.5
GCN           16.7           10.0             25.7
GIN           29.0           40.2            -10.5
  atom degree             23.3             16.1             30.6

  no_polycyclic: 1 of 3 architectures separated from the baseline after correction
    * GIN: +0.442 [+0.304, +0.588]  p = 0.000
      GAT: +0.058 [-0.036, +0.155]  p = 0.462
      GCN: -0.112 [-0.236, +0.020]  p = 0.287
  polycyclic_only: 1 of 3 architectures separated from the baseline after correction
    * GIN: -0.231 [-0.325, -0.144]  p = 0.000
      GAT: -0.062 [-0.135, +0.006]  p = 0.287
      GCN: -0.027 [-0.107, +0.051]  p = 0.488

Written to /content/drive/MyDrive/ames_pipeline/results
  - table76_regime_paired.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>